In [1]:
import numpy as np
import torch
from torch.linalg import solve_triangular

In [101]:
class GRF:
    def __init__(self, N_samples, l, device):
        super().__init__()
        self.device = device
        self.N_samples = N_samples #Number of GRF samples
        self.l = torch.tensor(l, device=self.device) #Length scale of GRF
        self.d = len(l) #Dimensionality of GRF
        self.n = tuple(((1.0 / self.l).ceil().long() + 1).tolist())
        self.mus = self.get_mus()
        self.coeffs = self.compute_coeffs()
        
    def get_mus(self):
        mus = ()
        for i in range(self.d):
            mus += (torch.linspace(0,1, self.n[i], device=self.device),)
        return mus

    def compute_covariances(self):
        K_i = ()
        for i in range(self.d):
            scaled_mus = self.mus[i] / self.l[i]
            diff = scaled_mus[:, None] - scaled_mus[None, :]
            dist_sq = diff**2
            K_i += (torch.exp(-0.5 * dist_sq),)
        return K_i
    
    def compute_coeffs(self):
        K_i = self.compute_covariances()
        L_i = tuple(torch.tensor(np.linalg.cholesky(K)) for K in K_i)
        Z = torch.randn(size=(self.N_samples,*self.n))
        f = Z
        for i in range(self.d):
            f = torch.tensordot(f, L_i[i], dims=([1],[0]))
        f = torch.tensor(f)
        y = f
        # Forward solves
        for i in range(self.d):
            y = torch.moveaxis(y, 1, -1)
            y = torch.linalg.solve_triangular(L_i[i], y, upper=False, left=False)
        #Backward solves
        f_m = y
        for i in range(self.d):
            f_m = torch.moveaxis(f_m, 1, -1)
            f_m = torch.linalg.solve_triangular(L_i[i].T, f_m, upper=True, left=False)
        return f_m        


In [105]:
grf = GRF(N_samples=100, l=(0.01, 0.01, 0.01), device='cuda')

AssertionError: Torch not compiled with CUDA enabled

In [111]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128


^C


In [43]:
A1 = torch.randn(3, 3).tril_()
A2 = torch.randn(4, 4).tril_()
A3 = torch.randn(4, 4).tril_()

B = torch.randn(10, 3, 4, 4)
print(B.shape)
axes = [0] + list(range(B.ndim - 1, 0, -1))
B = B.permute(*axes)
print(B.shape)
X = torch.linalg.solve_triangular(A1, B, upper=False, left=False)

X @ A1 - B

torch.Size([10, 3, 4, 4])
torch.Size([10, 4, 4, 3])


tensor([[[[ 0.0000e+00,  0.0000e+00,  0.0000e+00],
          [-1.4901e-08,  0.0000e+00,  0.0000e+00],
          [-1.1921e-07,  0.0000e+00,  0.0000e+00],
          [ 0.0000e+00, -2.9802e-08,  1.1921e-07]],

         [[-1.4901e-08,  0.0000e+00,  0.0000e+00],
          [ 0.0000e+00, -1.4901e-08,  0.0000e+00],
          [ 1.1921e-07,  2.3842e-07,  2.9802e-08],
          [ 0.0000e+00,  5.9605e-08,  5.9605e-08]],

         [[ 0.0000e+00,  0.0000e+00,  0.0000e+00],
          [ 2.9802e-08,  0.0000e+00,  0.0000e+00],
          [ 7.0781e-08,  0.0000e+00,  2.9802e-08],
          [-8.9407e-08,  0.0000e+00,  0.0000e+00]],

         [[ 0.0000e+00,  0.0000e+00,  0.0000e+00],
          [ 5.9605e-08,  0.0000e+00,  0.0000e+00],
          [ 5.9605e-08,  1.1921e-07,  0.0000e+00],
          [-1.4901e-08, -1.4901e-08,  0.0000e+00]]],


        [[[ 1.1921e-07,  0.0000e+00,  0.0000e+00],
          [ 2.9802e-08,  0.0000e+00,  0.0000e+00],
          [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
          [ 0.0000e+0